### Oskar Goluda s490083

# Sprawozdanie projektu: Przewidywanie ryzyka Udaru za pomocą modelu TabNet

## Spis Treści:
### 1. Przedstawienie tematu
### 2. Cele projektu 
### 3. Opis problemu
### 4. Dane wejściowe i wyjściowe
### 5. Opis algorytmu Tabelarycznej Sieci Neuronowej - TabNet
### 6. Opis algorytmu Regresji Logarytmicznej
### 7. Wyniki i wnioski Modelu TabNet do przewidywania Udaru 
### 8. Porównanie zaprezentowanych Metod

## 1. Przedstawienie tematu

Udar mózgu jest jedną z głównych przyczyn niepełnosprawności i śmiertelności na świecie. Wczesne wykrywanie ryzyka wystąpienia udaru na podstawie parametrów zdrowotnych i stylu życia może kluczowo wpłynąć na prewencję i szybką interwencję medyczną.

Niniejszy projekt koncentruje się na zastosowaniu technik uczenia maszynowego do analizy danych medycznych i przewidywania prawdopodobieństwa wystąpienia udaru. Głównym narzędziem modelowania jest **TabNet** – nowoczesna architektura sieci neuronowej zaprojektowana specjalnie do wydajnej pracy z danymi tabelarycznymi. Dla celów porównawczych wykorzystamy również klasyczną **Regresję Logistyczną**, oraz porównamy wyniki.

### Zbiór danych
Projekt wykorzystuje zbiór danych `healthcare-dataset-stroke-data.csv` udostępniony na platformie kaggle, który zawiera informacje o pacjentach, takie jak:
*   **Dane demograficzne:** Płeć, wiek, stan cywilny, typ zamieszkania.
*   **Dane medyczne:** Nadciśnienie, choroby serca, poziom glukozy, BMI.
*   **Styl życia:** Rodzaj pracy, status palenia tytoniu.

Zbiór ten zostanie wykorzystany do wyuczenia sieci neuronowej przewidującej ryzyko wystąpienia Udaru w oparciu o paramety. Największą przeszkodą będzie zauważalne w tym zbiorze niezrównoważenie klas - niedobór przypadów klasy 1: udaru.

## 2. Cele projektu

Głównym celem projektu jest stworzenie skutecznego narzędzia wspomagającego diagnozę ryzyka udaru. Zrealizowano to poprzez następujące cele szczegółowe:

1.  **Analiza i przygotowanie danych (obsugiwane przez skrypty z pliku data_preparation.py):**
    *   Czyszczenie danych (obsługa brakujących wartości, np. BMI).
    *   Inżynieria cech (Feature Engineering): Stworzenie nowych zmiennych takich jak grupy wiekowe (`age_group`), kategorie BMI (`bmi_category`) czy sumaryczny licznik czynników zwiększonego ryzyka (`risk_factor_count`).
    *   Transformacja danych: Kodowanie zmiennych kategorycznych i skalowanie zmiennych numerycznych.

2.  **Modelowanie z użyciem TabNet (plik train.py):**
    *   Implementacja i trening modelu TabNet, który łączy zalety drzew decyzyjnych (interpretowalność) z mocą głębokich sieci neuronowych.
    *   Dobór odpowiednich hiperparametrów (np. `n_steps`, `width`, `lambda_sparse`).

3.  **Ewaluacja i Porównanie:**
    *   Ocena jakości modelu za pomocą metryk takich jak: Accuracy, Precision, Recall, F1-Score oraz ROC-AUC.
    *   Porównanie wyników TabNet z modelem referencyjnym (Regresją Logistyczną) w celu weryfikacji skuteczności bardziej złożonego podejścia.

4.  **Interpretowalność i Wizualizacja:**
    *   Analiza ważności cech (Feature Importance) w celu zrozumienia, które czynniki mają największy wpływ na ryzyko udaru.
    *   Wizualizacja wyników za pomocą macierzy pomyłek (Confusion Matrix) oraz krzywej ROC.

## 3. Opis problemu klasyfikacji

Definicja zadania
W ramach projektu rozwiązujemy problem binarnej klasyfikacji nadzorowanej. Zadaniem modelu jest przypisanie każdemu pacjentowi jednej z dwóch rozłącznych etykiet oznaczających wystąpienie udaru ($1$) lub jego brak ($0$) na podstawie zestawu parametrów medycznych i demograficznych.

Sformułowanie matematyczne
Niech $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^N$ będzie zbiorem danych treningowych składającym się z $N$ próbek, gdzie:
* $x_i \in \mathbb{R}^d$ to $d$-wymiarowy wektor cech (input) opisujący $i$-tego pacjenta.
* $y_i \in \{0, 1\}$ to rzeczywista etykieta klasy (target).

Celem procesu uczenia jest znalezienie funkcji (modelu) $f_\theta: \mathbb{R}^d \to [0, 1]$, sparametryzowanej przez wektor wag $\theta$, która dla danej próbki $x_i$ estymuje prawdopodobieństwo warunkowe przynależności do klasy pozytywnej:
$$ \hat{y}_i = f_\theta(x_i) = P(y_i = 1 | x_i) $$

Ostateczna predykcja klasy $\hat{c}_i$ odbywa się poprzez zastosowanie progu decyzyjnego $t$:
$$ \hat{c}_i = \begin{cases} 1 & \text{dla } \hat{y}_i \ge t \\ 0 & \text{dla } \hat{y}_i < t \end{cases} $$

Optymalizacja i oczekiwany wynik
Oczekiwanym wynikiem jest minimalizacja błędu predykcji na danych testowych.

Minimalizacja tej funkcji metodą spadku gradientu pozwala modelowi TabNet na naukę nieliniowych zależności w danych tabelarycznych przy jednoczesnym zachowaniu rzadkości wybieranych cech.

##   4. Dane wejściowe i wyjściowe

Źródło i format:
Dane pochodzą z platformy Kaggle i zostały umieszczone w  pliku healthcare-dataset-stroke-data.csv umieszczonego w katalogu data/raw/. Jest to plik w formacie CSV, zawierający rekordy medyczne pacjentów.

Struktura cech :
Zbiór danych zawiera następujące zmienne wejściowe (cechy):

| Nazwa cechy | Typ | Opis |
|:--- |:--- |:--- |
| **gender** | Słowna | Płeć pacjenta (Male, Female, Other). |
| **age** | Numeryczna | Wiek pacjenta. |
| **hypertension** | Binarna | 0: brak nadciśnienia, 1: stwierdzone nadciśnienie. |
| **heart_disease** | Binarna | 0: brak chorób serca, 1: stwierdzone choroby serca. |
| **ever_married** | Słowna | Czy pacjent kiedykolwiek był w małżeństwie (No, Yes). |
| **work_type** | Słowna | Rodzaj pracy (children, Govt_job, Private, Self-employed, Never_worked). |
| **Residence_type** | Słowna | Miejsce zamieszkania (Rural, Urban). |
| **avg_glucose_level** | Numeryczna | Średni poziom glukozy we krwi. |
| **bmi** | Numeryczna | Wskaźnik masy ciała (Body Mass Index). |
| **smoking_status** | Słowna | Status palenia (formerly smoked, never smoked, smokes, Unknown). |


Zmienna celu:
* stroke – Zmienna binarna przybierająca wartość 1, jeśli pacjent przeszedł udar, oraz 0, jeśli nie.

Podział danych (Data Split):
Dane zostały podzielone na trzy niezależne podzbiory w celu rzetelnej ewaluacji modelu Zastosowano podział warstwowy, aby zachować identyczne proporcje przypadków udaru (klasy 1) w każdym ze zbiorów, co jest kluczowe przy niezbalansowanych danych.
<br>
1. Zbiór treningowy (70%): Służy do trenowania modelu.
2. Zbiór walidacyjny (15%): Używany do monitorowania postępów uczenia i tuningu hiperparametrów (Early Stopping).
3. Zbiór testowy (15%): Wykorzystywany wyłącznie do ostatecznej oceny skuteczności modelu i stworzenia metryk oceniających jego działanie.

### Sposób przygotowania danych (Replikacja wyników)

Aby zreplikować wyniki eksperymentu, zaimplementowano w projekcie automatyczny pipeline do przetwarzania danych, który wykonuje następujące kroki:

1. Czyszczenie danych:
* Usunięcie kolumny identyfikacyjnej id.
* Usunięcie wierszy z brakującymi wartościami w kolumnie bmi.
* Usunięcie błędnych rekordów (np. wiek <= 0).
2. Inżynieria cech (Feature Engineering):
* Dyskretyzacja zmiennych ciągłych: utworzenie grup wiekowych (age_group), kategorii BMI (bmi_category) oraz poziomów glukozy.
* Agregacja ryzyka: stworzenie zmiennej risk_factor_count sumującej występowanie nadciśnienia, chorób serca i palenia tytoniu.
3. Transformacja:
* Label Encoding: Zamiana zmiennych kategorycznych (tekstowych) na wartości liczbowe.
* Standard Scaling: Standaryzacja zmiennych numerycznych (age, bmi, avg_glucose_level) do średniej 0 i odchylenia standardowego 1.

Instrukcja uruchomienia:

Aby pobrać dane, przetworzyć je i uruchomić trening modelu (co automatycznie wygeneruje pliki w folderze data/processed oraz wyniki w results/), należy wykonać z poziomu głównego katalogu projektu polecenie:
```bash
python3 src/train.py
```

### Dane wejsciowe do predykcji

Dane wejsciowe są zapisane w pliku `example_input.csv` w formacie CSV Plik ten służy do przekazywania nowych, nieprzetworzonych danych o pacjentach do skryptu inferencyjnego (src/infer.py).

Format pliku:
Każdy wiersz reprezentuje pojedynczy przypadek medyczny, a kolumny muszą odpowiadać oryginalnym nazwom cech ze zbioru treningowego. Przykładowa zawartość pliku wygląda następująco:

```csv
gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
Female,54,1,0,Yes,Private,Urban,171.23,31.4,never smoked
Male,45,0,0,Yes,Self-employed,Rural,98.45,27.8,smokes
```

Po wykonaniu polecenia:
```bash
python3 src/infer.py --input example_input.csv
```
Dane te zostają przetworzone tak jak w przypadku danych ze zbiorów treningowego, walidacyjnego i testowego.Czyli są one poddane transformacji: Label Encoding'gowi, Scaling'owi oraz Feature Engineering'owi.

Tak przygotowany tensor przekazuwany jest do modelu TabNet w celu uzyskania predykcji,

### Dane wyjsciowe po predykcji

Po wykonaniu powyższego polecenia system generuje dane wyjściowe w dwóch formach:

1. Podsumowanie w terminalu.
Skrypt wyświetla w konsoli tabelaryczne zestawienie dla każdego pacjenta z pliku wejściowego.
Zawiera ono:
* Prawdopodobieństwo: Wyliczone przez model TabNet prawdopodobieństwo wystąpienia udaru (wartość od 0 do 1).
* Predykcja: Klasyfikacja binarna (1: Ryzyko udaru, 0: Brak ryzyka) na podstawie progu decyzyjnego.

2. Plik z wynikami `predictions.csv`:
W głównym katalogu projektu tworzony (lub nadpisywany) jest plik predictions.csv. Zawiera on kompletne wyniki, które można wykorzystać do dalszej analizy lub raportowania. 
Struktura pliku obejmuje:
* Wszystkie oryginalne kolumny wejściowe (dla łatwej identyfikacji pacjenta oraz możiwości dalszej analizy danych).
* Kolumnę stroke_probability – precyzyjny wynik pewności modelu.
* Kolumnę stroke_prediction – ostateczną decyzję klasyfikatora.
